# Create Datasets and Basic Summaries

In [368]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter
from datasets import load_dataset
from mapping import *
pd.set_option('display.max_columns', None)

In [5]:
path_raw = 'data/raw'

In [135]:
def summary_corpus(
    data: pd.DataFrame,
    column: str
    ):
    # Get length of text
    df = data.copy()
    df["length"] = df[column].apply(lambda x: len(x))

    # Unique words in corpus
    dict_words = dict(Counter(" ".join(df[column].values.tolist()).split(" ")).items())
    corpus = list(dict_words.keys())

    print(f"Records: {len(df)}")
    print(f"Average length: {round(np.average(df['length']), 2)}")
    print(f"Vocabulary: {len(corpus)}")

    return corpus

## 1. Affective Text

In [106]:
data_affectivetext = load_dataset("sem_eval_2018_task_1", 'subtask5.english')
dic_affectivetext = data_affectivetext.data
df_affectivetext = pd.concat([
                    dic_affectivetext["train"].to_pandas().assign(dataset = 'train'),
                    dic_affectivetext["validation"].to_pandas().assign(dataset = 'validation'),
                    dic_affectivetext["test"].to_pandas().assign(dataset = 'test')
                ])
df_affectivetext.head()

,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust,dataset
0,2017-En-21441,“Worry is a down payment on a problem you may ...,False,True,False,False,False,False,True,False,False,False,True,train
1,2017-En-31535,Whatever you decide to do make sure it makes y...,False,False,False,False,True,True,True,False,False,False,False,train
2,2017-En-21068,@Max_Kellerman it also helps that the majorit...,True,False,True,False,True,False,True,False,False,False,False,train
3,2017-En-31436,Accept the challenges so that you can literall...,False,False,False,False,True,False,True,False,False,False,False,train
4,2017-En-22195,My roommate: it's okay that we can't spell bec...,True,False,True,False,False,False,False,False,False,False,False,train


In [111]:
corpus_affectivetext = summary_corpus(df_affectivetext, 'Tweet')

Records: 10983
Average length: 94.79
Vocabulary: 37378


In [343]:
# Standard format
df_affectivetext_processed = df_affectivetext.copy()
df_affectivetext_processed.rename(columns={'ID':'id', 'Tweet':'text'}, inplace=True)
df_affectivetext_processed['emotion'] = df_affectivetext_processed.apply(lambda x: [col for col in df_affectivetext_processed.columns if x[col] == True], axis=1)
df_affectivetext_processed['platform'] = 'twitter'


df_affectivetext_processed.head()

,id,text,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust,dataset,emotion,platform
0,2017-En-21441,“Worry is a down payment on a problem you may ...,False,True,False,False,False,False,True,False,False,False,True,train,"[anticipation, optimism, trust]",twitter
1,2017-En-31535,Whatever you decide to do make sure it makes y...,False,False,False,False,True,True,True,False,False,False,False,train,"[joy, love, optimism]",twitter
2,2017-En-21068,@Max_Kellerman it also helps that the majorit...,True,False,True,False,True,False,True,False,False,False,False,train,"[anger, disgust, joy, optimism]",twitter
3,2017-En-31436,Accept the challenges so that you can literall...,False,False,False,False,True,False,True,False,False,False,False,train,"[joy, optimism]",twitter
4,2017-En-22195,My roommate: it's okay that we can't spell bec...,True,False,True,False,False,False,False,False,False,False,False,train,"[anger, disgust]",twitter


## 2. Crowdflower

In [112]:
path_crowdflower = "crowdflower"
df_crowdflower = pd.read_csv(f"{path_raw}/{path_crowdflower}/crowdflower.csv")
df_crowdflower.head()

,tweet_id,sentiment,author,content
0,1956967341,empty,xoshayzers,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,wannamama,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,coolfunky,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,czareaquino,wants to hang out with friends SOON!
4,1956968416,neutral,xkilljoyx,@dannycastillo We want to trade with someone w...


In [113]:
corpus_crowdflower = summary_corpus(df_crowdflower, 'content')

Records: 40000
Average length: 73.41
Vocabulary: 83294


In [367]:
df_crowdflower_processed = df_crowdflower.copy()

# Standard format
df_crowdflower_processed.rename(columns={'tweet_id':'id', 'content':'text', 'sentiment': 'emotion'}, inplace=True)
for col in df_crowdflower_processed['emotion'].unique():
    df_crowdflower_processed[col] = df_crowdflower_processed['emotion'].apply(lambda x: True if x == col else False)
df_crowdflower_processed['emotion'] = df_crowdflower_processed.apply(lambda x: [col for col in df_crowdflower_processed.columns if x[col] == True], axis=1)

# Remove empty
df_crowdflower_processed = df_crowdflower_processed[df_crowdflower_processed['empty'] == False]
df_crowdflower_processed.drop(columns=['author', 'empty'], inplace=True)

df_crowdflower_processed.head()

,id,emotion,text,sadness,enthusiasm,neutral,worry,surprise,love,fun,hate,happiness,boredom,relief,anger
1,1956967666,[sadness],Layin n bed with a headache ughhhh...waitin o...,True,False,False,False,False,False,False,False,False,False,False,False
2,1956967696,[sadness],Funeral ceremony...gloomy friday...,True,False,False,False,False,False,False,False,False,False,False,False
3,1956967789,[enthusiasm],wants to hang out with friends SOON!,False,True,False,False,False,False,False,False,False,False,False,False
4,1956968416,[neutral],@dannycastillo We want to trade with someone w...,False,False,True,False,False,False,False,False,False,False,False,False
5,1956968477,[worry],Re-pinging @ghostridah14: why didn't you go to...,False,False,False,True,False,False,False,False,False,False,False,False


## 3. Electoral Tweets

In [37]:
path_electoraltweets = 'Annotated-US2012-Election-Tweets/Questionnaire'
# Questionnaire 1
df_electoraltweets1 = pd.read_csv(f"{path_raw}/{path_electoraltweets}1/AnnotatedTweets.txt", sep='	')
# Questionnaire 2
df_electoraltweets2_1 = pd.read_csv(f"{path_raw}/{path_electoraltweets}2/Batch1/AnnotatedTweets.txt", sep='	')
df_electoraltweets2_2 = pd.read_csv(f"{path_raw}/{path_electoraltweets}2/Batch2/AnnotatedTweets.txt", sep='	')
# Merge
df_electoraltweets = pd.concat([df_electoraltweets2_1, df_electoraltweets2_2])
df_electoraltweets.head()

,unitid,createdat,golden,id,missed,startedat,tainted,channel,trust,workerid,country,region,city,tweet,q1whoisfeelingorwhofeltanemotioninotherwordswhoisthesourceoftheemotion,q2whatemotionchooseoneoftheoptionsfrombelowthatbestrepresentstheemotion,fontcolorolivetweetertweetfontbrq3ifthereisabetterwordfordescribingtheemotionthantheoneslistedabovethentypeithere,q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq4thenpleasetellusiftheemotioninthistweetispositivenegativeorneither,fontcolorolivetweetertweetfontbrq5howstronglyistheemotionbeingexpressedinthistweet,q6towardswhomorwhatinotherwordswhoorwhatisthetargetoftheemotion,fontcolorolivetweetertweetfontbrq7whichwordsinthetweethelpidentifyingtheemotion,q8whatreasoncanbededucedfromthetweetfortheemotionwhatisthecauseoftheemotion,fontcolorolivetweetertweetfontbrq9thistweetisaboutwhichofthefollowingissues,fontcolorolivetweetertweetfontbrq10ifthetweetisaboutanissuenotlistedabovethentypeithere,q11whichofthefollowingbestdescribesthepurposeofthistweet,origgolden,origq11whichofthefollowingbestdescribesthepurposeofthistweet,q11whichofthefollowingbestdescribesthepurposeofthistweetgold,q11whichofthefollowingbestdescribesthepurposeofthistweetgoldreason,fontcolorgreycommentsfont,q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq3thenpleasetellusiftheemotioninthistweetispositivenegativeorneither,q6bwhichofthesebestdescribesthetargetoftheemotion
0,234318245,12/17/2012 183705,True,775142387,BLANK,12/17/2012 181543,False,amt,1.0,11338794,USA,GA,Covington,"I'm a #Republican, but if I have to hear my mo...",Tweeter,disgust,Sarcasm,negative emotion,the emotion is being expressed with medium int...,my mom,I'm going to stab myself with my bayonet.,I have to hear my mom talk about #Romney,"About the election process, election publicity...",BLANK,"to point out hypocrisy, to disagree, to ridicu...",1,BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,NaN,NaN,NaN
1,234318378,12/16/2012 025145,False,772925098,BLANK,12/16/2012 022248,False,amt,1.0,11338794,USA,GA,Covington,Will Obama fire the person responsible for thi...,Tweeter,anger or annoyance or hostility or fury,BLANK,negative emotion,the emotion is being expressed with a high int...,not specified,"fire misguided individuals, hurt",hurt,None of the above,BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN
2,234318670,12/17/2012 185740,False,775168524,BLANK,12/17/2012 183815,False,amt,1.0,11338794,USA,GA,Covington,haha @DickMorrisTweet Romney is going to have ...,Tweeter,anger or annoyance or hostility or fury,belittle,neither positive nor negative,the emotion is being expressed with medium int...,DickMorrisTweet,haha,Romney is going to have a great convention. It...,"About the election process, election publicity...",BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN
3,234318253,12/17/2012 183705,False,775142384,BLANK,12/17/2012 181543,False,amt,1.0,11338794,USA,GA,Covington,S/0 to my newest @freeboosieRS &amp vote for O...,Tweeter,anticipation or expectancy or interest,BLANK,BLANK,the emotion is being expressed with medium int...,my newest @freeboosieRS,L0L,vote for Obama & boosie will soon to be free,About Social and Civil Issues but not related ...,BLANK,"to motivate, to entertain, or to provide infor...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN
4,234318724,12/17/2012 185740,False,775168523,BLANK,12/17/2012 183815,False,amt,1.0,11338794,USA,GA,Covington,Nicki Minaj Fucked Up With That Mitt Romney Li...,Tweeter,anger or annoyance or hostility or fury,BLANK,negative emotion,the emotion is being expressed with a high int...,Nicki Minaj,Fucked Up,Mitt Romney Line,"About the election process, election publicity...",BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN


In [114]:
corpus_electoraltweets = summary_corpus(df_electoraltweets, 'tweet')

Records: 4056
Average length: 102.56
Vocabulary: 6695


In [347]:
df_electoraltweets_processed = df_electoraltweets.copy()

# Standard format
keep_cols = ['id', 'tweet',
            'q2whatemotionchooseoneoftheoptionsfrombelowthatbestrepresentstheemotion',
            'q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq4thenpleasetellusiftheemotioninthistweetispositivenegativeorneither',
            'fontcolorolivetweetertweetfontbrq5howstronglyistheemotionbeingexpressedinthistweet'
            ]
df_electoraltweets_processed = df_electoraltweets_processed[keep_cols]
df_electoraltweets_processed\
    .rename(columns={   'q2whatemotionchooseoneoftheoptionsfrombelowthatbestrepresentstheemotion':'emotion',
                        'q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq4thenpleasetellusiftheemotioninthistweetispositivenegativeorneither': 'sentiment',
                        'fontcolorolivetweetertweetfontbrq5howstronglyistheemotionbeingexpressedinthistweet': 'intensity'
                    },
            inplace=True
            )
df_electoraltweets_processed['sentiment'] = df_electoraltweets_processed['sentiment'].apply(lambda x: re.sub(r' emotion| positive nor negative', '', x) if type(x)==str else 'BLANK')
df_electoraltweets_processed['intensity'] = df_electoraltweets_processed['intensity'].apply(lambda x: re.sub(r'.* with a |.* with | intensity', '', x))

# Pivot emotions
df_electoraltweets_processed['emotion'] = df_electoraltweets_processed['emotion'].apply(lambda x: re.sub(r'\s{2,}', ' ', x))
df_electoraltweets_processed['emotion_'] = df_electoraltweets_processed['emotion'].apply(lambda x: re.sub(r'\s+', '_', x))
df_electoraltweets_processed = df_electoraltweets_processed[df_electoraltweets_processed['emotion'] != 'BLANK']
for col in df_electoraltweets_processed['emotion_'].unique():
    df_electoraltweets_processed[col] = df_electoraltweets_processed['emotion_'].apply(lambda x: True if x == col else False)
df_electoraltweets_processed['emotion'] = df_electoraltweets_processed.apply(lambda x: [col for col in df_electoraltweets_processed.columns if x[col] == True], axis=1)

df_electoraltweets_processed.drop(columns=['emotion_'], inplace=True)
df_electoraltweets_processed.rename(columns={'tweet':'text'}, inplace=True)

df_electoraltweets_processed.head()

,id,text,emotion,sentiment,intensity,disgust,anger_or_annoyance_or_hostility_or_fury,anticipation_or_expectancy_or_interest,admiration,dislike,uncertainty_or_indecision_or_confusion,joy_or_happiness_or_elation,like,indifference,disappointment,calmness_or_serenity,hate,amazement,acceptance,fear_or_apprehension_or_panic_or_terror,vigilance,trust,surprise,sadness_or_gloominess_or_grief_or_sorrow
0,775142387,"I'm a #Republican, but if I have to hear my mo...",[disgust],negative,medium,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,772925098,Will Obama fire the person responsible for thi...,[anger_or_annoyance_or_hostility_or_fury],negative,high,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,775168524,haha @DickMorrisTweet Romney is going to have ...,[anger_or_annoyance_or_hostility_or_fury],neither,medium,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,775142384,S/0 to my newest @freeboosieRS &amp vote for O...,[anticipation_or_expectancy_or_interest],BLANK,medium,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,775168523,Nicki Minaj Fucked Up With That Mitt Romney Li...,[anger_or_annoyance_or_hostility_or_fury],negative,high,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


# 4. EmoInt

In [59]:
path_emoint_dev = 'EmoInt/EmoInt Dev'
path_emoint_test = 'EmoInt/EmoInt Test Gold'
path_emoint_train = 'EmoInt/EmoInt Train'
# Load all
l_emotions = ['anger', 'fear', 'joy', 'sadness']
l_emoint = []
for path_emoint, dataset in zip([path_emoint_dev, path_emoint_test, path_emoint_train], ['dev.target', 'test.gold', 'train']):
    for emotion in l_emotions:
        df = pd.read_table(f"{path_raw}/{path_emoint} Data_{emotion}-ratings-0to1.{dataset}.txt", 
                            delimiter='	',
                            names=['id', 'tweet', 'emotion', 'NONE'])
        df['dataset'] = dataset
        l_emoint.append(df)
# Merge
df_emoint = pd.concat(l_emoint).drop(columns=['NONE'])
df_emoint.head()

,id,tweet,emotion,dataset
0,10857,@ZubairSabirPTI pls dont insult the word 'Molna',anger,dev.target
1,10858,@ArcticFantasy I would have almost took offens...,anger,dev.target
2,10859,@IllinoisLoyalty that Rutgers game was an abom...,anger,dev.target
3,10860,@CozanGaming that's what lisa asked before she...,anger,dev.target
4,10861,Sometimes I get mad over something so minuscul...,anger,dev.target


In [115]:
corpus_emoint = summary_corpus(df_emoint, 'tweet')

Records: 7102
Average length: 95.25
Vocabulary: 24535


In [366]:
# Standard Format
df_emoint_processed = df_emoint.copy()
df_emoint_processed.rename({'tweet': 'text'}, inplace=True)
for col in df_emoint_processed['emotion'].unique():
    df_emoint_processed[col] = df_emoint_processed['emotion'].apply(lambda x: True if x == col else False)
df_emoint_processed['emotion'] = df_emoint_processed.apply(lambda x: [col for col in df_emoint_processed.columns if x[col] == True], axis=1)
df_emoint_processed['disgust'], df_emoint_processed['surprise'] = False, False

df_emoint_processed.rename(columns={'tweet':'text'}, inplace=True)

df_emoint_processed.head()

,id,text,emotion,dataset,anger,fear,joy,sadness,disgust,surprise
0,10857,@ZubairSabirPTI pls dont insult the word 'Molna',[anger],dev.target,True,False,False,False,False,False
1,10858,@ArcticFantasy I would have almost took offens...,[anger],dev.target,True,False,False,False,False,False
2,10859,@IllinoisLoyalty that Rutgers game was an abom...,[anger],dev.target,True,False,False,False,False,False
3,10860,@CozanGaming that's what lisa asked before she...,[anger],dev.target,True,False,False,False,False,False
4,10861,Sometimes I get mad over something so minuscul...,[anger],dev.target,True,False,False,False,False,False


# 5. Go Emotions

In [335]:
path_goemotions = 'goemotions'
# Load
data_goemotions = load_dataset("go_emotions")
dic_goemotions = data_goemotions.data
df_goemotions = pd.concat([
                    dic_goemotions["train"].to_pandas().assign(dataset = 'train'),
                    dic_goemotions["validation"].to_pandas().assign(dataset = 'validation'),
                    dic_goemotions["test"].to_pandas().assign(dataset = 'test')
                ])

In [336]:
corpus_goemotions = summary_corpus(df_goemotions, 'text')

Records: 54263
Average length: 68.33
Vocabulary: 65524


In [337]:
# Load Mappings
with open(f"{path_raw}/{path_goemotions}/emotions.txt", "r") as file:
    taxonomy = file.read().split("\n")

with open(f"{path_raw}/{path_goemotions}/ekman_mapping.json", "r") as file:
    mapping = json.load(file)

mapping |= {'neutral': ['neutral']}
taxonomy_ekman = list(mapping.keys())

# Mapping function
def labelEncoding(
    input: pd.DataFrame
    ) -> pd.DataFrame:

    df = input

    # Defining a function that maps each index to emotion labels
    def idx2class(idx_list):
        arr = []
        for i in idx_list:
            taxonomy_value = taxonomy[int(i)]
            ekman_value = [k for k, v in mapping.items() if taxonomy_value in v][0]
            arr.append(ekman_value)
        return arr

    # Applying the function
    df['emotions'] = df['labels'].apply(idx2class)

    # OneHot encoding for multi-label classification
    for emo in taxonomy_ekman:
        df[emo] = np.zeros((len(df),1))
        df[emo] = df['emotions'].apply(lambda x: True if emo in x else False)

    # Keeping only necessary columns
    #df = df.drop(['labels', 'emotions', 'neutral'], axis=1)
    df = df.loc[df[['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']].sum(axis=1) != 0]

    return df

# Encode
df_goemotions = labelEncoding(df_goemotions)
df_goemotions.head()

,text,labels,id,dataset,emotions,anger,disgust,fear,joy,sadness,surprise,neutral
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj,train,[anger],True,False,False,False,False,False,False
3,To make her feel threatened,[14],ed7ypvh,train,[fear],False,False,True,False,False,False,False
4,Dirty Southern Wankers,[3],ed0bdzj,train,[anger],True,False,False,False,False,False,False
5,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,[26],edvnz26,train,[surprise],False,False,False,False,False,True,False
6,Yes I heard abt the f bombs! That has to be wh...,[15],ee3b6wu,train,[joy],False,False,False,True,False,False,False


## 6. SSEC

In [354]:
path_ssec = 'ssec/ssec-aggregated'
threshold = '0.99'
df_ssec = pd.concat([
            pd.read_csv(f"{path_raw}/{path_ssec}/test-combined-{threshold}.csv", 
                        sep='\t', 
                        names=['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust', 'tweet'])\
                .assign(dataset='test'),
            pd.read_csv(f"{path_raw}/{path_ssec}/train-combined-{threshold}.csv", 
                        sep='\t', 
                        names=['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust', 'tweet'])\
                .assign(dataset='train'),
            ])
df_ssec['tweet'] = df_ssec['tweet'].astype(str)
df_ssec.head()

,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,tweet,dataset
0,---,---,---,---,---,---,---,---,He who exalts himself shall be humbled; a...,test
1,---,---,---,---,---,---,---,Trust,RT @prayerbullets: I remove Nehushtan -previou...,test
2,---,---,---,---,---,---,---,Trust,@Brainman365 @heidtjj @BenjaminLives I have so...,test
3,---,---,---,---,---,---,---,---,#God is utterly powerless without Human interv...,test
4,---,---,---,---,---,---,---,---,@David_Cameron Miracles of #Multiculturalism...,test


In [139]:
corpus_ssec = summary_corpus(df_ssec, 'tweet')

Records: 4870
Average length: 108.18
Vocabulary: 21082


In [363]:
df_ssec_processed = df_ssec.copy()
df_ssec_processed.reset_index(inplace=True)
for col in [i for i in df_ssec_processed.columns if i not in ['index', 'tweet', 'dataset']]:
    df_ssec_processed[col] = df_ssec_processed[col].apply(lambda x: True if x != '---' else False)
df_ssec_processed['emotion'] = df_ssec_processed.apply(lambda x: [col for col in df_ssec_processed.columns if x[col] == True], axis=1)
df_ssec_processed['noemotion'] = np.where(
                                    (df_ssec_processed['anger'] == False) &
                                    (df_ssec_processed['anticipation'] == False) &
                                    (df_ssec_processed['disgust'] == False) &
                                    (df_ssec_processed['fear'] == False) &
                                    (df_ssec_processed['joy'] == False) &
                                    (df_ssec_processed['sadness'] == False) &
                                    (df_ssec_processed['surprise'] == False) &
                                    (df_ssec_processed['trust'] == False), 1, 0)

df_ssec_processed.rename({'index': 'id', 'tweet': 'text'}, inplace=True)
df_ssec_processed = df_ssec_processed.loc[df_ssec_processed['noemotion'] == 0, df_ssec_processed.columns[:-1]]

df_ssec_processed.head()

,index,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,tweet,dataset,emotion
1,1,False,False,False,False,False,False,False,True,RT @prayerbullets: I remove Nehushtan -previou...,test,"[index, trust]"
2,2,False,False,False,False,False,False,False,True,@Brainman365 @heidtjj @BenjaminLives I have so...,test,[trust]
5,5,False,False,False,False,True,False,False,True,This world needs a tight group hug. Tight enou...,test,"[joy, trust]"
7,7,False,False,False,False,False,False,False,True,A Godly husband - knows you - trusts you - lo...,test,[trust]
9,9,True,False,False,False,False,False,False,False,#BIBLE = Big Irrelevant Book of Lies and Exagg...,test,[anger]


## 7. TEC

In [2]:
# Load Twitter Emotion Corpus
TEC_url='https://archive.org/download/misc-dataset/TEC.csv'
df_TEC=pd.read_csv(TEC_url)
df_TEC.head()

,tid,text,emotion
0,145353048817012736,Thinks that @melbahughes had a great 50th birt...,surprise
1,144279638024257536,"Como una expresión tan simple, una sola oració...",sadness
2,140499585285111809,the moment when you get another follower and y...,joy
3,145207578270507009,Be the greatest dancer of your life! practice ...,joy
4,139502146390470656,eww.. my moms starting to make her annual rum ...,disgust


In [140]:
corpus_TEC = summary_corpus(df_TEC, 'text')

Records: 21051
Average length: 84.71
Vocabulary: 55501


In [364]:
# Standard Format
df_TEC_processed = df_TEC.copy()
df_TEC_processed.rename(columns={'tid': 'id'}, inplace=True)
for col in df_TEC_processed['emotion'].unique():
    df_TEC_processed[col] = df_TEC_processed['emotion'].apply(lambda x: True if x == col else False)
df_TEC_processed['emotion'] = df_TEC_processed.apply(lambda x: [col for col in df_TEC_processed.columns if x[col] == True], axis=1)

df_TEC_processed.head()

,id,text,emotion,surprise,sadness,joy,disgust,fear,anger
0,145353048817012736,Thinks that @melbahughes had a great 50th birt...,[surprise],True,False,False,False,False,False
1,144279638024257536,"Como una expresión tan simple, una sola oració...",[sadness],False,True,False,False,False,False
2,140499585285111809,the moment when you get another follower and y...,[joy],False,False,True,False,False,False
3,145207578270507009,Be the greatest dancer of your life! practice ...,[joy],False,False,True,False,False,False
4,139502146390470656,eww.. my moms starting to make her annual rum ...,[disgust],False,False,False,True,False,False


## Convert to Ekman

In [384]:
def toEkman(
    data: pd.DataFrame,
    mapping: dict
    ) -> pd.DataFrame:

    df = data.copy()

    for emotion in mapping.keys():
        for emotype in mapping[emotion].keys():
            df[emotype + '_' + emotion] = np.where((df[mapping[emotion][emotype]]==True).any(axis=1), True, False)

    return df

In [385]:
toEkman(df_affectivetext_processed, map_affectivetext)

,id,text,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust,dataset,emotion,platform,primary_anger,secondary_anger,primary_disgust,secondary_disgust,primary_fear,secondary_fear,primary_joy,secondary_joy,primary_sadness,secondary_sadness,primary_surprise,secondary_surprise
0,2017-En-21441,“Worry is a down payment on a problem you may ...,False,True,False,False,False,False,True,False,False,False,True,train,"[anticipation, optimism, trust]",twitter,False,False,False,False,False,False,False,True,False,False,False,True
1,2017-En-31535,Whatever you decide to do make sure it makes y...,False,False,False,False,True,True,True,False,False,False,False,train,"[joy, love, optimism]",twitter,False,False,False,False,False,False,True,True,False,False,False,False
2,2017-En-21068,@Max_Kellerman it also helps that the majorit...,True,False,True,False,True,False,True,False,False,False,False,train,"[anger, disgust, joy, optimism]",twitter,True,False,True,False,False,False,True,True,False,False,False,False
3,2017-En-31436,Accept the challenges so that you can literall...,False,False,False,False,True,False,True,False,False,False,False,train,"[joy, optimism]",twitter,False,False,False,False,False,False,True,True,False,False,False,False
4,2017-En-22195,My roommate: it's okay that we can't spell bec...,True,False,True,False,False,False,False,False,False,False,False,train,"[anger, disgust]",twitter,True,False,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3254,2018-En-03848,shaft abrasions from panties merely shifted to...,True,False,False,False,False,False,False,True,False,False,False,test,"[anger, pessimism]",twitter,True,False,False,False,False,False,False,False,False,True,False,False
3255,2018-En-00416,@lomadia heard of Remothered? Indie horror gam...,False,True,False,False,False,False,False,True,False,False,False,test,"[anticipation, pessimism]",twitter,False,False,False,False,False,False,False,True,False,True,False,False
3256,2018-En-03717,All this fake outrage. Y'all need to stop 🤣,True,False,True,False,False,False,False,False,False,False,False,test,"[anger, disgust]",twitter,True,False,True,False,False,False,False,False,False,False,False,False
3257,2018-En-03504,Would be ever so grateful if you could record ...,False,False,False,False,True,False,False,False,False,False,False,test,[joy],twitter,False,False,False,False,False,False,True,False,False,False,False,False
